# Submission — BanglaT5 seed 42, decode → `submission.csv`

## ⚠️ Before running
| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (P100 is sm_60 and cannot run this) |
| Internet | **On** |
| Inputs | competition · `farhanishraqq/nascenia-code` · notebook **`nascenia-banglat5-v2`** |

The checkpoint comes from the training notebook's **output**. If that run was
interactive, do `File → Save Version → Quick Save` on it first — otherwise
`/kaggle/working` was never persisted and cell 3 will fail with instructions.

## What this does
1. **Validates on dev first** — beam vs MBR on 300 held-out rows
2. **Picks the winner automatically** on Token F1 / ROUGE-L
3. Generates `submission.csv` (`id,output`) on the 1,000 test rows using it

It does not blindly submit MBR output. MBR is expected to win — TRAIN-01 scored
Token F1 **0.2051**, within 0.0004 of TF-IDF retrieval and *below* a constant
string (0.2669), which says generating confident specifics is being penalised.
MBR picks the consensus candidate and discards risky specifics. **But that is a
hypothesis, and this notebook tests it before spending a submission slot.**

| Bar | Token F1 | Predicted LB |
|---|---|---|
| TRAIN-01 beam-4 | 0.2051 | 0.5574 |
| Constant string *(currently #1)* | 0.2669 | 0.57849 |
| Frequency-only constant | 0.3519 | — |

`LB ≈ 0.4672 + 0.3·TokenF1 + 0.2·ROUGE-L` — reproduces submission 1 exactly.

In [ ]:
# ══ 1 — hardware gate ═══════════════════════════════════════════════════════
import torch
n = torch.cuda.device_count()
assert n > 0, "No GPU. Settings -> Accelerator -> GPU T4 x2"
cap = torch.cuda.get_device_capability(0)
print(f"torch {torch.__version__} | {[torch.cuda.get_device_name(i) for i in range(n)]} | sm_{cap[0]}{cap[1]}")
assert cap[0] >= 7, f"WRONG ACCELERATOR sm_{cap[0]}{cap[1]} — use GPU T4 x2"
print("✅ hardware OK")

In [ ]:
# ══ 2 — pinned libs (trap #00: Kaggle ships transformers 5.0.0, which breaks T5) ══
!pip install -q --upgrade "transformers==4.57.3" git+https://github.com/csebuetnlp/normalizer
import transformers
assert transformers.__version__ == "4.57.3", transformers.__version__
from normalizer import normalize
print("transformers", transformers.__version__, "| normalizer OK:", normalize("হেলো,  নাসেনিয়া ডকে"))

In [ ]:
# ══ 3 — locate code, data, checkpoint ═══════════════════════════════════════
import glob, os, shutil, sys
print("/kaggle/input:", os.listdir("/kaggle/input"))

hits = glob.glob("/kaggle/input/**/04_decode.py", recursive=True)
assert hits, "Attach Add Input -> Datasets -> farhanishraqq/nascenia-code"
CODE = os.path.dirname(hits[0])

raw = glob.glob("/kaggle/input/**/test.csv", recursive=True)
assert raw, "Attach the Nascenia AI Hackathon competition (or a nascenia-data dataset)"
RAW = os.path.dirname(raw[0])

# checkpoint = any attached dir with config.json next to weights, excluding the code dataset
cands = [os.path.dirname(c) for c in glob.glob("/kaggle/input/**/config.json", recursive=True)
         if glob.glob(os.path.dirname(c) + "/*.safetensors") or glob.glob(os.path.dirname(c) + "/*.bin")]
assert cands, (
    "\n*** NO CHECKPOINT ***\n"
    "The training run's /kaggle/working was never persisted.\n"
    "On nascenia-banglat5-v2: File -> Save Version -> QUICK SAVE\n"
    "  (NOT 'Save & Run All' — that re-runs from scratch and destroys the weights)\n"
    "then here: Add Input -> Notebooks -> nascenia-banglat5-v2\n"
    f"searched: {os.listdir('/kaggle/input')}")
CKPT = sorted(cands, key=lambda p: ('best' not in p, len(p)))[0]

os.makedirs("/kaggle/working/code", exist_ok=True)
for f in glob.glob(f"{CODE}/*.py"):
    shutil.copy(f, "/kaggle/working/code/")
sys.path.insert(0, "/kaggle/working/code")
print("CODE:", CODE, "\nRAW :", RAW, "\nCKPT:", CKPT)
print("  ", sorted(os.listdir(CKPT))[:8])

In [ ]:
# ══ 4 — rebuild the identical frozen dev split (seed 42 — never change) ═════
!cd /kaggle/working/code && python 01_prep.py --raw "{RAW}" --out /kaggle/working/processed --seed 42 --dev-size 5000

## 5 — Validate on dev: beam vs MBR, same 300 rows
Decides which decoder writes the submission. ~20 min.

In [ ]:
# ══ 5a — beam baseline on 300 dev rows ══════════════════════════════════════
# subprocess rather than a multi-line `!shell \` continuation — the same pattern
# the training notebooks used successfully for long-running commands.
import subprocess, shlex

DEV = f"python 04_decode.py --ckpt {shlex.quote(CKPT)} --split dev --limit 300 --min-new-tokens 80 --no-bertscore"

cmd = DEV + " --mode beam --num-beams 4 --record /kaggle/working/dev_beam.json"
print(cmd, flush=True)
r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code")
assert r.returncode == 0, "beam decode failed"

In [ ]:
# ══ 5b — MBR on the SAME 300 dev rows ═══════════════════════════════════════
# 24 sampled candidates; utility = 0.3*TokenF1 + 0.2*ROUGE-L against the other samples.
# Same rows as 5a, so the comparison is like-for-like.
cmd = DEV + " --mode mbr -n 24 --temperature 0.8 --top-p 0.95 --record /kaggle/working/dev_mbr.json"
print(cmd, flush=True)
r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code")
assert r.returncode == 0, "mbr decode failed"

In [ ]:
# ══ 6 — pick the winner on lexical score (the only calibrated part of the metric) ══
import json

def lex(p):
    d = json.load(open(p))["dev"]
    return 0.3 * d["token_f1"] + 0.2 * d["rouge_l"], d["token_f1"], d["rouge_l"], d["mean_pred_tokens"]

b = lex("/kaggle/working/dev_beam.json")
m = lex("/kaggle/working/dev_mbr.json")
print(f"{'decoder':8s} {'TokenF1':>9s} {'ROUGE-L':>9s} {'lexical':>9s} {'tokens':>7s} {'pred LB':>9s}")
for name, v in (("beam", b), ("mbr", m)):
    print(f"{name:8s} {v[1]:9.4f} {v[2]:9.4f} {v[0]:9.4f} {v[3]:7.1f} {0.4672 + v[0]:9.4f}")

WINNER = "mbr" if m[0] > b[0] else "beam"
print(f"\n=> WINNER: {WINNER}   (Δ lexical {m[0]-b[0]:+.4f})")
print(f"   vs TRAIN-01 TokenF1 0.2051 | constant 0.2669 | frequency-only 0.3519")
if max(b[1], m[1]) < 0.2669:
    print("\n   ⚠️  STILL BELOW THE CONSTANT STRING — this submission would rank lower")
    print("      than the 0.57849 already on the board. Worth submitting only as a")
    print("      dev<->LB calibration point for model output; not as a ranking play.")

## 7 — Generate the submission on the 1,000 test rows
Uses whichever decoder won above.

In [ ]:
import subprocess, shlex

common = (f"python 04_decode.py --ckpt {shlex.quote(CKPT)} --split test "
          f"--min-new-tokens 80 --out /kaggle/working/submission.csv "
          f"--record /kaggle/working/test_run.json")
cmd = (common + " --mode mbr -n 24 --temperature 0.8 --top-p 0.95") if WINNER == "mbr" \
      else (common + " --mode beam --num-beams 4")

print(cmd + "\n" + "=" * 70, flush=True)
r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code")
assert r.returncode == 0, "decode failed"

In [ ]:
# ══ 8 — sanity checks (a malformed submission wastes a slot) ════════════════
import pandas as pd, glob, os

sub = pd.read_csv("/kaggle/working/submission.csv")
test = pd.read_csv(glob.glob("/kaggle/input/**/test.csv", recursive=True)[0])

problems = []
if len(sub) != 1000:                       problems.append(f"expected 1000 rows, got {len(sub)}")
if list(sub.columns) != ["id", "output"]:  problems.append(f"columns are {list(sub.columns)}, must be ['id','output']")
if sub["id"].duplicated().any():           problems.append("duplicate ids")
if set(sub["id"]) != set(test["id"]):      problems.append("id set does not match test.csv")
if sub["output"].isna().any():             problems.append("null outputs")
if (sub["output"].astype(str).str.strip() == "").any(): problems.append("empty outputs")

print(f"rows {len(sub)} | cols {list(sub.columns)} | unique ids {sub['id'].nunique()}")
print(f"mean output length {sub['output'].astype(str).str.split().str.len().mean():.1f} tokens (reference ~100)")
print("\n❌ " + "; ".join(problems) if problems else "\n✅ all checks passed — submission.csv ready")
sub.head(3)

---
## After this run

**Submit from the Output tab yourself** — `submission.csv`.

Then record in `PREDICTIONS.md`: predicted LB from cell 6, the actual score, and the delta.
That delta is the point — the `LB ≈ 0.4672 + 0.3·F1 + 0.2·RL` formula is currently
calibrated on a **constant string only**. This is the first chance to check it holds for
**model-generated** output, and every downstream decision leans on it.

Archive the run as `NOTEBOOKS/(score)_(notebook_name)/` per CLAUDE.md.